# Task 041: phasorpy-main — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: FLIM phasor analysis for fluorescence lifetime imaging

**Paper**: Software library (cite via Zenodo)
**Repository**: https://github.com/phasorpy/phasorpy

### Physical Background

Physical systems have parameters (frequencies, damping, material constants) extracted by fitting models to measurements.

### Forward Model

```
y = f(theta) + n
```

### Inverse Problem

```
theta_hat = argmin ||y - f(theta)||^2  or Bayesian inference
```

### Performance Metrics
- **PSNR**: 25.78 dB
- **SSIM**: N/A


### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import os
import sys
import math
import logging
import numbers
import warnings
import numpy as np
import scipy.ndimage
from scipy.ndimage import median_filter
import tifffile
import xarray
import pooch

# =============================================================================
# Utils and Helper Functions (Must be defined first)
# =============================================================================

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`parse_signal_axis`, `parse_harmonic`, `fetch`, `phasor_filter_median_impl`, `phasor_threshold_impl`, `load_and_preprocess_data`

In [ ]:
def parse_signal_axis(signal, axis=None):
    """Identifies the signal axis for processing."""
    if hasattr(signal, 'dims'):
        if axis is None:
            for ax in 'HC':
                if ax in signal.dims:
                    return signal.dims.index(ax)
            return -1
        if isinstance(axis, int):
            return axis
        if axis in signal.dims:
            return signal.dims.index(axis)
        raise ValueError(f'{axis=} not found in {signal.dims!r}')
    if axis is None:
        return -1
    if isinstance(axis, int):
        return axis
    raise ValueError(f'invalid {axis=} for {type(signal)=}')

def parse_harmonic(harmonic, harmonic_max=None):
    """Parses harmonic input into a list of integers."""
    if harmonic_max is not None and harmonic_max < 1:
        raise ValueError(f'{harmonic_max=} < 1')

    if harmonic is None:
        return [1], False

    if isinstance(harmonic, (int, numbers.Integral)):
        if harmonic < 1 or (harmonic_max is not None and harmonic > harmonic_max):
            raise IndexError(f'{harmonic=!r} is out of bounds [1, {harmonic_max}]')
        return [int(harmonic)], False

    if isinstance(harmonic, str):
        if harmonic == 'all':
            if harmonic_max is None:
                raise TypeError(f'maximum harmonic must be specified for {harmonic=!r}')
            return list(range(1, harmonic_max + 1)), True
        raise ValueError(f'invalid {harmonic=!r}')

    h = np.atleast_1d(harmonic)
    if h.size == 0:
        raise ValueError(f'{harmonic=!r} is empty')
    return [int(i) for i in harmonic], True

def fetch(fname):
    """Fetches data from remote or local source."""
    if os.path.exists(fname):
        return os.path.abspath(fname)
        
    script_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
    file_path = os.path.join(script_dir, fname)
    if os.path.exists(file_path):
        return file_path

    if fname == 'Embryo.tif':
        url = 'https://github.com/phasorpy/phasorpy-data/raw/main/zenodo_8046636/Embryo.tif'
        file_hash = 'd1107de8d0f3da476e90bcb80ddf40231df343ed9f28340c873cf858ca869e20'
        return pooch.retrieve(
            url=url,
            known_hash='sha256:' + file_hash,
            fname=fname,
            path=pooch.os_cache('phasorpy'),
            progressbar=True
        )
    return fname

def phasor_filter_median_impl(mean, real, imag, size=3, repeat=1):
    """Median filter implementation for phasor coordinates."""
    mean = np.asarray(mean)
    real = np.asarray(real)
    imag = np.asarray(imag)
    
    if repeat == 0:
        return mean, real, imag
        
    for _ in range(repeat):
        real = median_filter(real, size=size)
        imag = median_filter(imag, size=size)
        
    return mean, real, imag

def phasor_threshold_impl(mean, real, imag, mean_min=0):
    """Thresholds phasor coordinates based on intensity."""
    mask = mean < mean_min
    
    mean_out = mean.copy()
    real_out = real.copy()
    imag_out = imag.copy()
    
    mean_out[mask] = np.nan
    real_out[mask] = np.nan
    imag_out[mask] = np.nan
    
    return mean_out, real_out, imag_out

def load_and_preprocess_data(filename, harmonic='all'):
    """
    Loads image data and determines dimensions.
    Returns the raw data array and preprocessing metadata.
    """
    local_path = fetch(filename)
    
    # Load using tifffile
    with tifffile.TiffFile(local_path) as tif:
        data = tif.asarray()
        
        # Heuristic for Embryo.tif or general timelapse FLIM data
        # Embryo.tif is typically (Time, Y, X) = (56, 128, 128)
        shape = data.shape
        dims = list('CYX') # Default guess
        
        if len(shape) == 3:
            # Assume (Time/Harmonic, Y, X)
            dims = ['H', 'Y', 'X'] 
        elif len(shape) == 4:
            dims = ['C', 'H', 'Y', 'X']
            
        # Basic normalization or type conversion if needed
        data = data.astype(np.float32)

    return data, {'dims': dims, 'shape': shape, 'source': local_path}

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
y = f(theta) + n
```

Functions: `forward_operator`

In [ ]:
def forward_operator(signal, axis=0, harmonic=1):
    """
    Computes the Forward Model: Signal (Time Domain) -> Phasor (Frequency Domain).
    Returns (mean, real, imag) components.
    """
    signal = np.asarray(signal)
    
    # Handle negative indexing or dimension matching
    ndim = signal.ndim
    if axis < 0:
        axis += ndim
        
    samples = signal.shape[axis]
    harmonic_list, has_harmonic_axis = parse_harmonic(harmonic, samples // 2)
    
    # Prepare for FFT: move signal axis to last
    if axis != ndim - 1:
        signal_swapped = np.swapaxes(signal, axis, -1)
    else:
        signal_swapped = signal

    # Real FFT
    # signal_swapped shape: (..., samples)
    fft_values = np.fft.rfft(signal_swapped, axis=-1)
    
    # Extract DC (0th component)
    dc = fft_values[..., 0].real
    
    # Avoid division by zero
    valid_dc = np.abs(dc) > 1e-9
    
    means = dc / samples
    
    reals = []
    imags = []
    
    for h in harmonic_list:
        if h < fft_values.shape[-1]:
            val = fft_values[..., h]
            
            # Normalization logic:
            # Phasor G (real) = Re(DFT) / DC
            # Phasor S (imag) = -Im(DFT) / DC  (Note the sign flip for standard phasor definition)
            
            r = np.zeros_like(dc)
            i = np.zeros_like(dc)
            
            r[valid_dc] = val.real[valid_dc] / dc[valid_dc]
            i[valid_dc] = -val.imag[valid_dc] / dc[valid_dc]
            
            reals.append(r)
            imags.append(i)
        else:
            raise ValueError(f"Harmonic {h} too high for samples {samples}")
            
    # Return formatted arrays
    if len(harmonic_list) == 1 and not has_harmonic_axis:
        return means, reals[0], imags[0]
    else:
        return means, np.stack(reals, axis=0), np.stack(imags, axis=0)

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
theta_hat = argmin ||y - f(theta)||^2  or Bayesian inference
```

Functions: `run_inversion`

In [ ]:
def run_inversion(mean, real, imag, original_axis=0, original_samples=None, harmonic=None, filter_params=None):
    """
    Performs the Inverse Model: Phasor -> Signal (Reconstruction).
    Includes optional filtering step on the phasor coordinates before inversion.
    """
    # 1. Apply Filtering (if params provided)
    if filter_params:
        mean_f, real_f, imag_f = phasor_filter_median_impl(
            mean, real, imag, 
            size=filter_params.get('median_size', 3), 
            repeat=filter_params.get('median_repeat', 1)
        )
        mean_f, real_f, imag_f = phasor_threshold_impl(
            mean_f, real_f, imag_f, 
            mean_min=filter_params.get('threshold_min', 0)
        )
    else:
        mean_f, real_f, imag_f = mean, real, imag

    # 2. Prepare Reconstruction parameters
    mean_arr = np.asarray(mean_f)
    real_arr = np.asarray(real_f)
    imag_arr = np.asarray(imag_f)
    
    harmonic_list, _ = parse_harmonic(harmonic)
    
    if original_samples is None:
        # Default to Nyquist based on max harmonic if unknown
        original_samples = max(harmonic_list) * 2 + 1
        
    phase_grid = np.linspace(0, 2*np.pi, original_samples, endpoint=False)
    
    # 3. Reconstruction Logic: Signal = Mean * (1 + 2 * Sum(G*cos + S*sin))
    
    # Initialize signal with DC component
    # We need to broadcast mean_arr to (samples, ...)
    # Assume mean_arr shape is (Y, X) or similar spatial dims
    
    # Create broadcast shape
    # If mean is (Y, X), result is (samples, Y, X) initially
    target_shape = (original_samples,) + mean_arr.shape
    rec_signal = np.ones(target_shape) * mean_arr[np.newaxis, ...]
    
    # Iterate through harmonics to add modulation
    # Check if input arrays have a harmonic axis
    has_harmonic_dim = (real_arr.ndim == mean_arr.ndim + 1)
    
    if has_harmonic_dim:
        # real_arr is (H, Y, X)
        for h_idx, h_val in enumerate(harmonic_list):
            r = real_arr[h_idx]
            i = imag_arr[h_idx]
            
            # Angle: (samples,)
            angle = phase_grid * h_val
            
            # Reshape angle for broadcasting: (samples, 1, 1)
            angle_reshaped = angle.reshape((-1,) + (1,) * (mean_arr.ndim))
            
            # r, i are (Y, X), broadcast to (1, Y, X)
            r_b = r[np.newaxis, ...]
            i_b = i[np.newaxis, ...]
            
            term = r_b * np.cos(angle_reshaped) + i_b * np.sin(angle_reshaped)
            rec_signal += 2 * mean_arr[np.newaxis, ...] * term
            
    else:
        # Scalar harmonic input
        h_val = harmonic_list[0]
        r = real_arr
        i = imag_arr
        
        angle = phase_grid * h_val
        angle_reshaped = angle.reshape((-1,) + (1,) * (mean_arr.ndim))
        
        r_b = r[np.newaxis, ...]
        i_b = i[np.newaxis, ...]
        
        term = r_b * np.cos(angle_reshaped) + i_b * np.sin(angle_reshaped)
        rec_signal += 2 * mean_arr[np.newaxis, ...] * term

    # 4. Move axis to original position if necessary
    if original_axis != 0:
        rec_signal = np.moveaxis(rec_signal, 0, original_axis)
        
    return rec_signal

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_results`

In [ ]:
def evaluate_results(original_data, reconstructed_data):
    """
    Computes error metrics between original and reconstructed data.
    """
    # Create mask for valid data (exclude NaNs from thresholding)
    mask = ~np.isnan(reconstructed_data) & ~np.isnan(original_data)
    
    if mask.sum() == 0:
        return {'mse': float('nan'), 'psnr': float('nan'), 'corr': float('nan')}
        
    orig_valid = original_data[mask]
    rec_valid = reconstructed_data[mask]
    
    # MSE
    mse = np.mean((orig_valid - rec_valid)**2)
    
    # PSNR
    max_val = np.max(orig_valid)
    if mse > 0:
        psnr = 10 * np.log10(max_val**2 / mse)
    else:
        psnr = float('inf')
        
    # Correlation Coefficient
    if orig_valid.size > 1:
        corr = np.corrcoef(orig_valid.flatten(), rec_valid.flatten())[0, 1]
    else:
        corr = 0.0
        
    return {'mse': mse, 'psnr': psnr, 'corr': corr}

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Configuration
DATA_FILE = 'Embryo.tif'
HARMONIC_NUM = 1
FILTER_CONFIG = {'median_size': 3, 'median_repeat': 1, 'threshold_min': 5.0}

### 1. Load Data

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 1. Load Data
print("--- Loading Data ---")
data, meta = load_and_preprocess_data(DATA_FILE)
print(f"Data Loaded: Shape {data.shape}, Dims {meta['dims']}")

# Identify time axis (axis 0 for this dataset based on shape (56, 128, 128))
time_axis = 0
num_samples = data.shape[time_axis]

### 2. Forward Operator (Phasor Transform)

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 2. Forward Operator (Phasor Transform)
print("--- Running Forward Operator ---")
mean_p, real_p, imag_p = forward_operator(data, axis=time_axis, harmonic=HARMONIC_NUM)
print(f"Phasor Mean Shape: {mean_p.shape}")

### 3. Run Inversion (Filtering + Reconstruction)

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 3. Run Inversion (Filtering + Reconstruction)
print("--- Running Inversion (Filter + Reconstruction) ---")
reconstructed_signal = run_inversion(
    mean_p, real_p, imag_p, 
    original_axis=time_axis, 
    original_samples=num_samples, 
    harmonic=HARMONIC_NUM,
    filter_params=FILTER_CONFIG
)
print(f"Reconstructed Shape: {reconstructed_signal.shape}")

### 4. Evaluate

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 4. Evaluate
print("--- Evaluating Results ---")
metrics = evaluate_results(data, reconstructed_signal)

print(f"MSE: {metrics['mse']:.4f}")
print(f"PSNR: {metrics['psnr']:.2f} dB")
print(f"Correlation: {metrics['corr']:.4f}")

print("OPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **phasorpy-main**:

1. **Problem Setup**: Physical systems have parameters (frequencies, damping, material constants) extracted by fitting models to measurements.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=25.78 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Software library (cite via Zenodo)
- Repository: https://github.com/phasorpy/phasorpy
